In [1]:
"""
Morris Sampling for PFLOTRAN Sensitivity Analysis
===================================================
Generates parameter combinations for Morris method using SALib.
All parameters are sampled in log10 space since ranges span orders of magnitude.
"""

import numpy as np
from SALib.sample import morris as morris_sample
from SALib import ProblemSpec

In [2]:
# =============================================================================
# Step 1: Define the problem
# =============================================================================
# SALib requires a 'problem' dictionary with parameter names and bounds.
# We define bounds in log10 space so that Morris discretizes uniformly
# across orders of magnitude.

problem = {
    'num_vars': 16,
    'names': [
        'calcite_rate',
        'fh_rmax',          # ferrihydrite DIR max rate
        'fh_k_donor',       # ferrihydrite DIR acetate half-sat
        'fh_SA',            # ferrihydrite surface area
        'gt_rmax',          # goethite DIR max rate
        'gt_k_donor',       # goethite DIR acetate half-sat
        'gt_SA',            # goethite surface area
        'root_resp',        # root respiration rate
        'msr_rmax',         # sulfate reduction max rate
        'msr_k_donor',      # sulfate reduction acetate half-sat
        'msr_k_acceptor',   # sulfate reduction sulfate half-sat
        'denit_rmax',       # denitrification max rate
        'denit_k_donor',    # denitrification acetate half-sat
        'denit_k_acceptor', # denitrification nitrate half-sat
        'aero_rate',        # aerobic respiration rate constant
        'aero_k_o2',        # aerobic respiration O2 half-sat
    ],
    # Bounds in log10 space
    'bounds': [
        [np.log10(1.55e-07), np.log10(1.55e-05)],   # calcite rate
        [np.log10(2.50e-07), np.log10(2.50e-05)],   # fh_rmax
        [np.log10(1.00e-07), np.log10(1.00e-05)],   # fh_k_donor
        [np.log10(1.08e+06), np.log10(1.08e+08)],   # fh_SA
        [np.log10(2.50e-08), np.log10(2.50e-06)],   # gt_rmax
        [np.log10(1.00e-07), np.log10(1.00e-05)],   # gt_k_donor
        [np.log10(2.83e+05), np.log10(2.83e+07)],   # gt_SA
        [np.log10(7.94e-14), np.log10(7.94e-12)],   # root_resp
        [np.log10(2.50e-08), np.log10(2.50e-06)],   # msr_rmax
        [np.log10(5.00e-07), np.log10(5.00e-05)],   # msr_k_donor
        [np.log10(1.00e-05), np.log10(1.00e-03)],   # msr_k_acceptor
        [np.log10(1.00e-07), np.log10(1.00e-05)],   # denit_rmax
        [np.log10(1.00e-06), np.log10(1.00e-04)],   # denit_k_donor
        [np.log10(1.00e-06), np.log10(1.00e-04)],   # denit_k_acceptor
        [np.log10(1.00e-10), np.log10(1.00e-08)],   # aero_rate
        [np.log10(1.00e-05), np.log10(1.00e-03)],   # aero_k_o2
    ]
}

In [3]:
# =============================================================================
# Step 2: Generate Morris trajectories
# =============================================================================
# N = number of trajectories (r in Morris notation)
# num_levels = number of discrete levels per parameter (p)
# optimal_trajectories = number of candidate trajectories to generate and
#     select from to maximize spread (set to None for pure random)

param_values_log = morris_sample.sample(
    problem,
    N=20,                       # 20 trajectories
    num_levels=4,               # 4 discrete levels per parameter
    optimal_trajectories=None,  # set to an integer to optimize (e.g., 10)
    seed=42                     # for reproducibility
)

print(f"Sample matrix shape: {param_values_log.shape}")
# Expected: (340, 16) — that is, 20 * (16+1) = 340 rows

Sample matrix shape: (340, 16)


In [7]:
# =============================================================================
# Step 3: Convert from log10 space back to real parameter values
# =============================================================================
# SALib sampled in log10 space, so we exponentiate to get actual values.

param_values_real = 10**param_values_log

print(f"\nFirst trajectory (17 runs):")
print(f"{'Run':<5}", end="")
for name in problem['names']:
    print(f"{name:>16}", end="")
print()
for i in range(17):
    print(f"{i:<5}", end="")
    for j in range(16):
        print(f"{param_values_real[i, j]:>16.4e}", end="")
    print()


First trajectory (17 runs):
Run      calcite_rate         fh_rmax      fh_k_donor           fh_SA         gt_rmax      gt_k_donor           gt_SA       root_resp        msr_rmax     msr_k_donor  msr_k_acceptor      denit_rmax   denit_k_donordenit_k_acceptor       aero_rate       aero_k_o2
0          7.1945e-07      2.5000e-05      2.1544e-06      5.0129e+06      2.5000e-08      4.6416e-07      2.8300e+07      7.9400e-14      2.5000e-08      5.0000e-05      2.1544e-04      1.0000e-05      1.0000e-04      4.6416e-06      4.6416e-10      2.1544e-04
1          7.1945e-07      2.5000e-05      2.1544e-06      5.0129e+06      2.5000e-08      4.6416e-07      1.3136e+06      7.9400e-14      2.5000e-08      5.0000e-05      2.1544e-04      1.0000e-05      1.0000e-04      4.6416e-06      4.6416e-10      2.1544e-04
2          7.1945e-07      2.5000e-05      2.1544e-06      5.0129e+06      2.5000e-08      4.6416e-07      1.3136e+06      7.9400e-14      2.5000e-08      5.0000e-05      2.1544e-04    

In [6]:
# =============================================================================
# Step 4: Save to CSV for downstream use
# =============================================================================
import pandas as pd

df = pd.DataFrame(param_values_real, columns=problem['names'])
df.index.name = 'run_id'
df.to_csv('/Users/christiandewey/Code/dewey-etal_meanders/sensitivity/morris_parameter_sets.csv')

print(f"\nSaved {len(df)} parameter sets to morris_parameter_sets.csv")


Saved 340 parameter sets to morris_parameter_sets.csv
